# Case A / 案例 A：Bank fraud classification / 银行欺诈分类
This case uses two fixed CSV files with the same schema and business meaning: `case-a-fraud-teaching.csv` contains S1–S8 for hand calculation, while `case-a-fraud.csv` contains 6,000 independently generated rows for model comparison. The generator verifies both files byte for byte. / 本案例使用两份字段语义一致的固定 CSV：`case-a-fraud-teaching.csv` 包含用于手算的 S1—S8，`case-a-fraud.csv` 包含独立生成的 6,000 条模型对照数据。Notebook 会逐字节核对两份文件。所有机制与数字均为教学设定。

In [ ]:
import numpy as np
import pandas as pd
import sklearn, xgboost, lightgbm, catboost
from scipy.special import expit
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from pathlib import Path
import sys
bundle = Path('content/posts/2026-10-17-from-a-single-tree-to-a-forest-mathematical-principles-algorithmic-evolution-and-engineering-practices-of-tree-models')
if not bundle.is_dir():
    bundle = Path.cwd()
sys.path.insert(0, str(bundle.resolve()))
from generate_tree_cases import generate_fraud, generate_fraud_teaching
teaching_path = bundle / 'case-a-fraud-teaching.csv'
csv_path = bundle / 'case-a-fraud.csv'
assert teaching_path.read_text(encoding='utf-8') == generate_fraud_teaching().to_csv(index=False, float_format='%.12g')
assert csv_path.read_text(encoding='utf-8') == generate_fraud().to_csv(index=False, float_format='%.12g')
toy = pd.read_csv(teaching_path).rename(columns={'record_id':'id'})
data = pd.read_csv(csv_path)
assert len(toy) == 8 and len(data) == 6000 and data.record_id.is_unique
print('Versions / 版本:', sklearn.__version__, xgboost.__version__, lightgbm.__version__, catboost.__version__)


## 1. Hand calculation / 手算
Here `g=p-y` is the gradient, not the residual. Net gain already subtracts gamma, so a split is favorable when net gain is positive. / 净增益已扣除 gamma，故只需与零比较。

In [ ]:
toy = toy.reset_index(drop=True)
p0, lam, gamma, eta = 0.5, 1.0, 0.1, 0.3
toy['g1'] = p0 - toy.y
toy['h1'] = p0 * (1 - p0)
def stats(frame):
    return float(frame.g1.sum()), float(frame.h1.sum())
def gain(parent, left, right):
    g, h = stats(parent); gl, hl = stats(left); gr, hr = stats(right)
    return 0.5 * (gl**2/(hl+lam) + gr**2/(hr+lam) - g**2/(h+lam)) - gamma
left = toy[toy.amount <= 350]; right = toy[toy.amount > 350]
ll = left[left.distance <= 160]; lr = left[left.distance > 160]
root_gain = gain(toy, left, right)
left_gain = gain(left, ll, lr)
assert len(toy) == 8 and toy.y.sum() == 4
assert np.isclose(root_gain, 1.0428571428571427)
assert np.isclose(left_gain, 0.5)
weights = {name: -stats(frame)[0]/(stats(frame)[1]+lam) for name, frame in [('LL', ll), ('LR', lr), ('R', right)]}
assert np.allclose(list(weights.values()), [-1.0, 0.4, 6/7])
toy['leaf'] = np.select([toy.id.isin(ll.id), toy.id.isin(lr.id)], ['LL', 'LR'], default='R')
toy['z1'] = eta * toy.leaf.map(weights)
toy['p1'] = expit(toy.z1)
toy['g2'] = toy.p1 - toy.y
toy['h2'] = toy.p1 * (1 - toy.p1)
retail = toy[toy.merchant_type=='retail']; digital = toy[toy.merchant_type=='digital']
h_retail = -(0.2*np.log2(0.2)+0.8*np.log2(0.8))
info_gain = 1-len(retail)/8*h_retail
split_info = -(5/8*np.log2(5/8)+3/8*np.log2(3/8))
assert len(retail)==5 and len(digital)==3 and retail.y.sum()==1 and digital.y.sum()==3
assert np.isclose(info_gain,0.5487949406953986) and np.isclose(info_gain/split_info,0.574995168878684)
assert np.isclose(0.5-5/8*(1-0.2**2-0.8**2),0.3)
print('Merchant type information gain / 商户类型信息增益:',round(info_gain,6))
print('Root net gain:', round(root_gain, 6), 'left-child net gain:', round(left_gain, 6))
print('Leaf weights:', weights)
print(toy[['id','amount','distance','merchant_type','y','g1','h1','leaf','p1','g2','h2']].round(4).to_string(index=False))


## 2. Ordered-statistic illustration / 有序统计示意
The following fixed order and smoothed mean illustrate self-label exclusion. They are **not** CatBoost's exact internal CTR values. / 固定顺序和平滑均值只解释排除自身标签的原则，并非 CatBoost 内部 CTR 的精确输出。

In [ ]:
prior, strength = 0.5, 1.0
seen = {}
enc = []
for row in toy.itertuples():
    total, count = seen.get(row.merchant_type, (0, 0))
    enc.append((total + strength*prior)/(count + strength))
    seen[row.merchant_type] = (total + row.y, count + 1)
toy['ordered_demo'] = enc
toy['full_mean_leaky'] = toy.groupby('merchant_type').y.transform('mean')
assert np.isclose(toy.loc[0, 'ordered_demo'], 0.5)
assert np.isclose(toy.loc[1, 'ordered_demo'], 0.25)
print(toy[['id','merchant_type','y','ordered_demo','full_mean_leaky']].to_string(index=False))


## 3. Full imbalanced CSV / 完整不平衡数据
Merchant identity has a stable, synthetic risk effect; its label is never used as a feature. / 商户身份带有设定的风险差异，但标签绝不作为特征。A stratified 60/20/20 split is held fixed across models. / 三模型共享分层划分。

In [ ]:
train = data.loc[data.split=='train'].copy()
valid = data.loc[data.split=='valid'].copy()
test = data.loc[data.split=='test'].copy()
assert toy.id.tolist() == [f'S{i}' for i in range(1,9)]
assert set(toy.id).isdisjoint(set(data.record_id))
assert 'teaching_row' not in data.columns
assert len(data)==6000 and len(train)==3600 and len(valid)==len(test)==1200
assert not set(train.index)&set(valid.index) and not set(train.index)&set(test.index)
print('Counts / 样本量:', len(train), len(valid), len(test), 'positive rate / 正类比例:', round(data.y.mean(),4))


## 4. Native-category model comparison / 原生类别模型对照
Only positive-class weight changes within each model pair. The 0.5 threshold is fixed; AP and ROC-AUC use probabilities. / 每组只改变正类权重；固定 0.5 阈值计算分类指标，AP 与 ROC-AUC 使用概率。These small synthetic results are not a general model ranking. / 结果不是普遍排名。

In [ ]:
feature_cols = ['amount','distance','merchant_type','merchant_id']
categories = {c: sorted(train[c].unique()) for c in ['merchant_type','merchant_id']}
def frame(part, family):
    x = part[feature_cols].copy()
    for col, values in categories.items():
        x[col] = x[col].astype(pd.CategoricalDtype(values)) if family!='CatBoost' else x[col].astype(str)
    return x
ratio = (train.y==0).sum()/(train.y==1).sum()
rows = []
for family in ['XGBoost','LightGBM','CatBoost']:
    for weighted in [False, True]:
        kwargs = {'scale_pos_weight':ratio} if weighted else {}
        if family=='XGBoost':
            model = XGBClassifier(n_estimators=100,max_depth=4,learning_rate=.06,tree_method='hist',enable_categorical=True,n_jobs=1,random_state=42,eval_metric='logloss',**kwargs)
        elif family=='LightGBM':
            model = LGBMClassifier(n_estimators=100,num_leaves=15,min_child_samples=20,learning_rate=.06,verbosity=-1,n_jobs=1,random_state=42,**kwargs)
        else:
            model = CatBoostClassifier(iterations=100,depth=4,learning_rate=.06,boosting_type='Ordered',loss_function='Logloss',cat_features=['merchant_type','merchant_id'],thread_count=1,random_seed=42,verbose=False,allow_writing_files=False,**kwargs)
        model.fit(frame(train,family),train.y)
        score = model.predict_proba(frame(test,family))[:,1]
        pred = score>=.5
        tp = int(((test.y==1)&pred).sum()); fp = int(((test.y==0)&pred).sum())
        fn = int(((test.y==1)&~pred).sum()); tn = int(((test.y==0)&~pred).sum())
        rows.append([family,'weighted' if weighted else 'base',tp,fp,fn,tn,precision_score(test.y,pred,zero_division=0),recall_score(test.y,pred,zero_division=0),f1_score(test.y,pred,zero_division=0),average_precision_score(test.y,score),roc_auc_score(test.y,score)])
results_a = pd.DataFrame(rows,columns=['model','setting','TP','FP','FN','TN','precision','recall','f1','AP','ROC_AUC'])
print(results_a.round(3).to_string(index=False))


## 5. LightGBM design diagnostics / LightGBM 设计示意
The one-hot columns below are mutually exclusive, so they illustrate what EFB can bundle; the tiny table cannot establish a speedup. GOSS is configured in a separate fit and is not the default in this version. / 下表只说明 EFB 的互斥条件；小数据不能证明提速。GOSS 另行显式启用。

In [ ]:
one_hot = pd.get_dummies(toy.merchant_type).astype(int)
assert (one_hot.sum(axis=1)==1).all()
print('Mutually exclusive / 逐行互斥:'); print(one_hot.to_string(index=False))
goss = LGBMClassifier(n_estimators=30,num_leaves=15,min_child_samples=20,learning_rate=.06,data_sample_strategy='goss',top_rate=.2,other_rate=.1,verbosity=-1,n_jobs=1,random_state=42)
goss.fit(frame(train,'LightGBM'),train.y)
print('GOSS fit completed / GOSS 训练完成:', goss.n_estimators_)
